# Notebook 03 — RFM Feature Engineering

## What this notebook does
Takes one clean row per delivered order from Silver and produces
one scored row per customer with Recency, Frequency, and Monetary
scores plus supporting signals.

## Why RFM
RFM is the industry-standard method for customer value segmentation.
Used at Adobe, Salesforce, Klaviyo, Twilio, and every e-commerce
analytics team. It answers: who are your best customers, who is
drifting, and who has already left.

## The scoring method: ntile(5)
Divides all customers into 5 equal buckets on each dimension.
Score 5 = best. Score 1 = worst.
Recency is inverted — fewer days since purchase = higher score.

## Source → Target
Source : brazilian.silver.customer_orders (managed Delta)
Target : brazilian.gold.customer_rfm_scores (managed Delta)

## Critical decision: snapshot date
We use MAX(purchase_ts) from the dataset, NOT today's date.
Olist data ends late 2018. Using today() makes every single
customer look like they churned 6+ years ago.
MAX(purchase_ts) = the reference point the dataset was built around.

In [0]:
# ── SETUP ─────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE CATALOG brazilian")

SOURCE_TABLE = "brazilian.silver.customer_orders"
TARGET_TABLE = "brazilian.gold.customer_rfm_scores"

print(f"Reading from : {SOURCE_TABLE}")
print(f"Writing to   : {TARGET_TABLE}")

In [0]:
# ── SNAPSHOT DATE — the most important decision in RFM ────────
#
# WHY NOT today():
# This dataset was collected in 2016-2018.
# If we calculate recency as DATEDIFF(today(), last_purchase_date),
# every customer would have 2000+ days of recency.
# Every single customer would score 1 on Recency.
# The entire RFM model collapses — no meaningful differentiation.
#
# WHY MAX(purchase_ts):
# We treat the last day any order was placed as "today"
# for the purposes of this analysis.
# Customers who ordered close to that date are Recent.
# Customers who ordered much earlier are Not Recent.
# This is how Olist themselves would have run this analysis
# if they had done it at the time the data was collected.
#
# INTERVIEW ANSWER when asked about this:
# "I used MAX(purchase_ts) as the snapshot date rather than
#  today's date to avoid look-ahead bias. The dataset ends in
#  late 2018 — using today() would make every customer appear
#  churned with 2000+ days of recency, collapsing the model.
#  In a production pipeline this would be parameterized as the
#  pipeline run date so it updates automatically each day."

snapshot_date = spark.sql(f"""
    SELECT MAX(purchase_ts) AS max_date
    FROM {SOURCE_TABLE}
""").collect()[0]["max_date"]

print(f"Snapshot date : {snapshot_date}")
print()
print("This is the reference point for all Recency calculations.")
print("DATEDIFF(snapshot_date, last_purchase_date) = recency_days")
print()
print("A customer whose last order was 10 days before snapshot → recency 10")
print("A customer whose last order was 300 days before snapshot → recency 300")
print("Lower recency_days = more recent = better = higher R score")

In [0]:
# ── COMPUTE RAW RFM METRICS ───────────────────────────────────
#
# WHY GROUP BY customer_unique_id:
# Silver has one row per ORDER.
# Gold needs one row per CUSTOMER.
# GROUP BY collapses all orders for one person into one summary row.
#
# RECENCY  = days since last purchase
#            DATEDIFF(snapshot, MAX(purchase_ts))
#            We use MAX because we want the MOST RECENT order date
#
# FREQUENCY = number of distinct completed orders
#             COUNT(DISTINCT order_id)
#             We use DISTINCT because the same order_id could theoretically
#             appear more than once if data has duplicates
#
# MONETARY  = total spend across all orders
#             SUM(payment_value)
#             payment_value in Silver is already the actual amount paid
#             (we aggregated installments in the Silver JOIN)
#
# EXTRA SIGNALS beyond standard RFM:
# avg_review_score   → satisfaction proxy, key churn predictor
#                      low review score + low recency = dissatisfied churner
# avg_items_per_order → basket size signal, indicates purchase depth
# customer_tenure_days → how long between first and last order
#                        tenure = 0 means only one order ever placed

rfm_raw = spark.sql(f"""
    SELECT
        customer_unique_id,

        -- RECENCY
        -- snapshot_date minus the most recent purchase date
        -- Result: how many days has this customer been silent
        DATEDIFF('{snapshot_date}', MAX(purchase_ts))        AS recency_days,

        -- FREQUENCY
        -- Count of distinct completed (delivered) orders per customer
        COUNT(DISTINCT order_id)                             AS frequency,

        -- MONETARY
        -- Total spend across all delivered orders
        -- This is lifetime value for the observation window
        ROUND(SUM(payment_value), 2)                         AS monetary_value,

        -- EXTRA SIGNAL 1: satisfaction
        -- Average review score across all orders this customer placed
        -- Null where customer never left a review — kept as null
        ROUND(AVG(review_score), 2)                          AS avg_review_score,

        -- EXTRA SIGNAL 2: basket size
        -- Average number of products per order
        -- High basket size = deeper product relationship
        ROUND(AVG(item_count), 2)                            AS avg_items_per_order,

        -- EXTRA SIGNAL 3: tenure
        -- Days between first and last order
        -- tenure = 0 → single-order customer (important for New segment)
        DATEDIFF(MAX(purchase_ts), MIN(purchase_ts))         AS customer_tenure_days,

        -- DATES for reference
        MIN(purchase_ts)                                     AS first_purchase_date,
        MAX(purchase_ts)                                     AS last_purchase_date

    FROM {SOURCE_TABLE}
    GROUP BY customer_unique_id
""")

row_count = rfm_raw.count()
print(f"Unique customers aggregated : {row_count:,}")
print()
print("RFM metric distributions:")
rfm_raw.describe([
    "recency_days",
    "frequency",
    "monetary_value",
    "avg_review_score"
]).show()

In [0]:
# ── RFM SCORING WITH ntile(5) WINDOW FUNCTION ─────────────────
#
# WHAT ntile(5) DOES:
# Sorts all customers by the metric, then divides them into
# 5 equal groups. Group 5 = top 20%, Group 1 = bottom 20%.
#
# WHY ntile() instead of fixed thresholds:
# Fixed thresholds like "recency < 30 days = score 5" break
# when the dataset changes — what if all customers bought
# within 30 days? Everyone gets score 5, no differentiation.
# ntile() is always relative — it ALWAYS produces a meaningful
# spread regardless of the actual values in the data.
# This is the industry standard method.
#
# RECENCY SCORING — INVERTED:
# Lower recency_days = bought more recently = BETTER
# So we ORDER BY recency_days ASC and subtract from 6
# Customer with recency 5 days  → ntile bucket 1 → 6-1 = score 5 ✅
# Customer with recency 400 days → ntile bucket 5 → 6-5 = score 1 ✅
#
# FREQUENCY SCORING — STANDARD:
# Higher frequency = more orders = BETTER
# ORDER BY frequency ASC → bucket 5 = most orders = score 5 ✅
#
# MONETARY SCORING — STANDARD:
# Higher monetary_value = more spend = BETTER
# ORDER BY monetary_value ASC → bucket 5 = highest spend = score 5 ✅
#
# rfm_total = r_score + f_score + m_score
# Range: 3 (worst: 1+1+1) to 15 (best: 5+5+5)
# Used in Notebook 04 to assign segment labels

rfm_scored = (rfm_raw

    # RECENCY SCORE: invert ntile result
    # Window ordered ASC (low days = recent = good)
    # Subtract from 6 to flip: bucket 1 → score 5, bucket 5 → score 1
    .withColumn("r_score",
        (6 - F.ntile(5).over(
            Window.orderBy(F.asc("recency_days"))
        )).cast("int")
    )

    # FREQUENCY SCORE: standard ntile
    # Window ordered ASC (high frequency = good = bucket 5 = score 5)
    .withColumn("f_score",
        F.ntile(5).over(
            Window.orderBy(F.asc("frequency"))
        ).cast("int")
    )

    # MONETARY SCORE: standard ntile
    # Window ordered ASC (high spend = good = bucket 5 = score 5)
    .withColumn("m_score",
        F.ntile(5).over(
            Window.orderBy(F.asc("monetary_value"))
        ).cast("int")
    )

    # TOTAL RFM SCORE: sum of all three scores
    # Range 3-15. Higher = better overall customer.
    # Used as tiebreaker and for segment boundary decisions.
    .withColumn("rfm_total",
        F.col("r_score") + F.col("f_score") + F.col("m_score")
    )
)

print("Scoring complete. Score distribution check:")
print("Each score bucket should have roughly equal customer counts.")
print()

# Verify each score 1-5 has similar customer counts
# If wildly unequal, something is wrong with the ntile logic
spark.sql("""
    SELECT r_score,
           COUNT(*) AS customers,
           ROUND(AVG(monetary_value), 2) AS avg_spend,
           ROUND(AVG(recency_days), 0)   AS avg_recency
    FROM rfm_scored_temp
    GROUP BY r_score
    ORDER BY r_score DESC
""") if False else None  # placeholder for temp view approach below

rfm_scored.createOrReplaceTempView("rfm_scored_temp")

print("R Score distribution (should be roughly equal buckets):")
spark.sql("""
    SELECT r_score,
           COUNT(*) AS customers,
           ROUND(AVG(recency_days), 0) AS avg_recency_days
    FROM rfm_scored_temp
    GROUP BY r_score ORDER BY r_score DESC
""").show()

print("F Score distribution:")
spark.sql("""
    SELECT f_score,
           COUNT(*) AS customers,
           ROUND(AVG(frequency), 2) AS avg_orders
    FROM rfm_scored_temp
    GROUP BY f_score ORDER BY f_score DESC
""").show()

print("M Score distribution:")
spark.sql("""
    SELECT m_score,
           COUNT(*) AS customers,
           ROUND(AVG(monetary_value), 2) AS avg_spend
    FROM rfm_scored_temp
    GROUP BY m_score ORDER BY m_score DESC
""").show()

In [0]:
# ── WRITE TO GOLD ─────────────────────────────────────────────
#
# WHY GOLD schema:
# This is the first Gold table — business-ready, scored data.
# Unlike Bronze (raw) and Silver (cleaned), Gold tables are
# directly used to make business decisions.
# A marketing team can query this table and immediately understand
# which customers are valuable and which are drifting.

(rfm_scored
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TARGET_TABLE))

written_count = spark.table(TARGET_TABLE).count()
print(f"✅ {TARGET_TABLE}")
print(f"   Rows written : {written_count:,} customers scored")

In [0]:
# ── POST-WRITE VALIDATION ─────────────────────────────────────
#
# Three checks:
# 1. No nulls in critical scoring columns
# 2. Scores are all within valid range 1-5
# 3. rfm_total is within valid range 3-15

print("POST-WRITE VALIDATION — brazilian.gold.customer_rfm_scores")
print()

# Check 1: Nulls in score columns
print("Null check on score columns (all must be 0):")
spark.sql(f"""
    SELECT
        SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END) AS null_uid,
        SUM(CASE WHEN r_score            IS NULL THEN 1 ELSE 0 END) AS null_r_score,
        SUM(CASE WHEN f_score            IS NULL THEN 1 ELSE 0 END) AS null_f_score,
        SUM(CASE WHEN m_score            IS NULL THEN 1 ELSE 0 END) AS null_m_score,
        SUM(CASE WHEN rfm_total          IS NULL THEN 1 ELSE 0 END) AS null_rfm_total
    FROM {TARGET_TABLE}
""").show()

# Check 2: Score range validation
# Every score must be between 1 and 5 inclusive
# If min < 1 or max > 5 the ntile logic has an error
print("Score range check (min must be 1, max must be 5):")
spark.sql(f"""
    SELECT
        MIN(r_score) AS r_min, MAX(r_score) AS r_max,
        MIN(f_score) AS f_min, MAX(f_score) AS f_max,
        MIN(m_score) AS m_min, MAX(m_score) AS m_max,
        MIN(rfm_total) AS total_min,
        MAX(rfm_total) AS total_max
    FROM {TARGET_TABLE}
""").show()

# Check 3: Sample top and bottom customers
print("Top 5 customers by rfm_total (best customers):")
spark.sql(f"""
    SELECT customer_unique_id,
           recency_days, frequency, monetary_value,
           r_score, f_score, m_score, rfm_total,
           avg_review_score
    FROM {TARGET_TABLE}
    ORDER BY rfm_total DESC
    LIMIT 5
""").show(truncate=False)

print("Bottom 5 customers by rfm_total (most at risk):")
spark.sql(f"""
    SELECT customer_unique_id,
           recency_days, frequency, monetary_value,
           r_score, f_score, m_score, rfm_total,
           avg_review_score
    FROM {TARGET_TABLE}
    ORDER BY rfm_total ASC
    LIMIT 5
""").show(truncate=False)

print()
print("=" * 65)
print("RFM COMPLETE")
print(f"  {TARGET_TABLE}")
print(f"  {written_count:,} customers scored on R, F, M (1-5 each)")
print(f"  rfm_total range: 3 (worst) to 15 (best)")
print("  Next: 04_churn_segments.py")
print("=" * 65)